#[4] Basic Agent

### API Key

In [ ]:
from dotenv import load_dotenv
import os
load_dotenv(dotenv_path="/content/.env", override=True)

print(".env 내 OPENAI_API_KEY가 환경변수에 할당됐습니다:", os.environ["OPENAI_API_KEY"][:5]+"*****")

### Install package

In [ ]:
!pip install -q langchain langchain-openai langchain-community

#1. Tool 사용

##Tool 없는 Agent

In [ ]:
from langchain.chat_models import init_chat_model

model = init_chat_model("gpt-5-nano")

In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model = model,
)

In [ ]:
agent

In [ ]:
response = agent.invoke(
    {"messages": [{"role": "user", "content": "현재 한국 날씨 어때?"}]},
)

In [ ]:
response["messages"][-1].pretty_print()

##1-1. 날씨

In [ ]:
from langchain.tools import tool

@tool
def get_weather(location : str) -> str :
  "이 도구는 특정 지역의 날씨 정보를 반환함."
  return f"오늘 {location} 날씨는 비가 옵니다."

In [ ]:
# from langchain.agents import create_agent

agent = create_agent(
    model = model,
    tools = [get_weather]
)

In [ ]:
agent

In [ ]:
agent.invoke(
    {"messages": [{"role": "user", "content": "오늘 서울 날씨 어때요? 날씨에 따라 옷 스타일을 추천해줘요"}]},
)

##1-2. 사칙연산

In [ ]:
from langchain.tools import tool

@tool
def add(a: int, b: int) -> int:
    """`a`와 `b` 덧셈.

    Args:
        a: First int
        b: Second int
    """
    return a + b

@tool
def substract(a: int, b: int) -> int:
    """`a`에서 `b`를 빼기.

    Args:
        a: First int
        b: Second int
    """
    return a - b

@tool
def multiply(a: int, b: int) -> int:
    """`a`와 `b` 곱셈.

    Args:
        a: First int
        b: Second int
    """
    return a * b

@tool
def divide(a: int, b: int) -> float:
    """`a`와 `b` 나눗셈.

    Args:
        a: First int
        b: Second int
    """
    return a / b

In [ ]:
tools = [add, substract, multiply, divide]

In [ ]:
agent = create_agent(
    model=model,
    tools = tools
)

In [ ]:
agent

In [ ]:
result = agent.invoke(
    {"messages": [{"role": "user", "content": "42 + 3 - 23은 뭔가요?"}]},
)

In [ ]:
result["messages"][-1].pretty_print()

In [ ]:
result

In [ ]:
agent = create_agent(
    model,
    tools,
    system_prompt="너는 사칙연산할 때 무조건 도구를 호출해야 해"
)

In [ ]:
result = agent.invoke(
    {"messages": [{"role": "user", "content": "42 + 3 * 23은 뭔가요?"}]},
)

In [ ]:
result

##1-3. 알라딘

In [ ]:
response = model.invoke("현재 알라딘에서 베스트셀러 Top10 뭐야?")

In [ ]:
response.content

알라딘 키 발급 (회원 가입 필요) :

https://blog.aladin.co.kr/openapi/popup/6695306


In [ ]:
from langchain.tools import tool
from typing import List, Dict, Any

import requests

@tool
def fetch_aladin_bestseller_topN(top_n:int) -> List[Dict[str, Any]]:
    """
    알라딘 베스트셀러 목록을 조회하고 Top N개(기본 1)를 반환합니다.

    Args:
        top_n: Top 개수 (값이 없을때는 1로하며, 10을 초과하는 경우 10으로 함)
    """
    url = "http://www.aladin.co.kr/ttb/api/ItemList.aspx"
    params = {
        "ttbkey": ".....",  # Key 입력
        "QueryType": "Bestseller",
        "MaxResults": top_n,
        "start": 1,
        "SearchTarget": "Book",
        "output": "js",
        "Version": "20131101"
    }
    resp = requests.get(url, params=params)
    resp.raise_for_status()
    data = resp.json()
    items = data.get("item", [])
    top10 = items[:10]
    return top10

In [ ]:
agent = create_agent(
    model,
    [fetch_aladin_bestseller_topN],
)

In [ ]:
agent

In [ ]:
response = agent.invoke(
    {"messages": [{"role": "user", "content": "현재 알라딘에서 베스트셀러 Top 5가 뭐야?"}]},
)

In [ ]:
response['messages'][-1].content

In [ ]:
for i, msg in enumerate(response["messages"]):
    print(f"--- Message {i+1} : {msg.type} ---")
    print(msg.content)
    print()

##1-4. 챗봇

In [ ]:
from langchain_core.tools import tool
import datetime
from datetime import timezone, timedelta

# 도구 정의
@tool
def get_current_time() -> str:
    """현재 시간을 반환합니다."""

    # KST는 UTC+9
    kst = timezone(timedelta(hours=9))
    now_kst = datetime.datetime.now(kst)
    return now_kst.strftime("%Y년 %m월 %d일 %H시 %M분")


In [ ]:
@tool
def calculate(expression: str) -> float:
    """수학 계산을 수행합니다.

    Args:
        expression: 계산할 수식 (예: "2 + 3 * 4")
    """
    try:
        result = eval(expression)
        return f"{expression} = {result}"
    except Exception as e:
        return f"계산 오류: {e}"


In [ ]:
@tool
def get_weather(city: str) -> str:
    """도시의 날씨를 조회합니다. 도시명은 한글로 해줘요

    Args:
        city: 도시 이름 (예: '서울', '부산', '제주')
    """
    # 실제로는 날씨 API 호출
    # 여기서는 예시 데이터 반환
    weather_data = {
        "서울": "맑음 ☀️ 22°C",
        "부산": "흐림 ☁️ 20°C",
        "제주": "비 🌧️ 18°C"
    }
    return weather_data.get(city, f"{city}의 날씨 정보를 찾을 수 없습니다.")


In [ ]:
# 도구 목록
tools = [get_current_time, calculate, get_weather]

In [ ]:
agent = create_agent(
    model=model,
    tools = tools,
    system_prompt="너는 도구를 확인하고, 적절한 도구를 호출해서 신중하게 답변해야 해"
)

In [ ]:
agent

In [ ]:
agent.invoke({"messages": [{"role": "user", "content": "지금 몇 시야?"}]})

In [ ]:
agent.invoke({"messages": [{"role": "user", "content": "15 곱하기 8은?"}]})

In [ ]:
agent.invoke({"messages": [{"role": "user", "content": "부산 날씨 알려줘"}]})

In [ ]:
# 대화형 질의
print("질문을 입력하세요 (종료: exit or quit)")
while True:
    question = input("\n질문: ")
    if question.lower() in ['exit', 'quit', '종료', '끝']:
        print("종료합니다.")
        break

    response = agent.invoke({"messages": [{"role": "user", "content": question}]})

    # print(f"답변: {response}")
    print(f"답변: {response["messages"][-1].content}")

#2. 단기Memory

##① Agent 생성 - checkpointer

In [ ]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model,
    tools,
    checkpointer = InMemorySaver(),
)

##② Agent 실행 - thread_id

In [ ]:
response = agent.invoke(
    {"messages": [
        {"role": "user",
         "content": "안녕하세요. 저는 Jumany입니다."}
        ]
     },
    {"configurable": {"thread_id": "thread_1"}},
)

In [ ]:
print(response["messages"][-1].content)

In [ ]:
response = agent.invoke(
    {"messages": [
        {"role": "user",
         "content": "안녕하세요. 제 이름이 뭐죠?"}
        ]
     },
    {"configurable": {"thread_id": "thread_1"}},
 )

In [ ]:
print(response["messages"][-1].content)

In [ ]:
response = agent.invoke(
    {"messages": [
        {"role": "user",
         "content": "안녕하세요. 제 이름이 뭐죠?"}
        ]
     },
    {"configurable": {"thread_id": "thread_2"}},
)

In [ ]:
print(response["messages"][-1].content)

In [ ]:
response = agent.invoke(
    {"messages": [{"role": "user", "content": "지금까지 무슨 얘기 나눴죠?"}]},
    {"configurable": {"thread_id": "thread_1"}},
)

In [ ]:
response

In [ ]:
for i, msg in enumerate(response["messages"]):
    print(f"--- Message {i+1} : {msg.type} ---")
    print(msg.content)
    print()

In [ ]:
response = agent.invoke(
    {"messages": [{"role": "user", "content": "지금까지 무슨 얘기 나눴죠?"}]},
    {"configurable": {"thread_id": "thread_2"}},
)

In [ ]:
for i, msg in enumerate(response["messages"]):
    print(f"--- Message {i+1} : {msg.type} ---")
    print(msg.content)
    print()

# 구조화된 답변

## ① Tool 정의

In [ ]:
from langchain.tools import tool
from typing import List, Dict

# 이메일 전송 도구
@tool
def send_email_tool(to: str, subject: str, body: str) -> str:
    """
    지정한 이메일 주소로 메일을 보내는 도구입니다.

    Args:
        to: 수신자 이메일 주소
        subject: 이메일 제목
        body: 이메일 본문 내용
    """
    return f"✅ 이메일이 성공적으로 전송되었습니다.\n수신자: {to}\n제목: {subject}\n내용: {body[:50]}..."


# 이메일 읽기 도구
@tool
def read_email_tool(limit: int = 3) -> str:
    """
    고객이 온라인 쇼핑몰에 보낸 컴플레인, 문의, 혹은 확인 관련 이메일을 읽는 도구입니다.
    """
    return f"✅ 이메일이 성공적으로 조회되었습니다."

##참고.**LLM Tool Emulator : '흉내' 내기**

에뮬레이터가 내놓는 결과는 언어 모델이 논리적으로 추론해 만들어낸 '가짜 데이터' 임

(예를들어, 연결해야 할 외부 API가 아직 개발 중이거나, 호출당 발생하는 비용이 너무 비싸 매번 실제 데이터를 가져오기 부담스러운 상황에 유용)

In [ ]:
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain.agents.middleware import LLMToolEmulator

agent = create_agent(
    model="gpt-5-nano",
    tools=[send_email_tool, read_email_tool],
    middleware=[
        LLMToolEmulator(model="gpt-5-nano"),
    ],
)

In [ ]:
response = agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "최근 온 메일 확인하고 고객의 의도와 감정, 요약, 그리고 어떤 행동이 필요한지 분석하세요."}
        ]
    }
)

In [ ]:
response["messages"][-1].pretty_print()

##② Structured output 정의

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal

# Structured Output 정의
class EmailAnalysis(BaseModel):
    """이메일 내용을 분석한 결과 구조."""
    intent: Literal["complaint", "inquiry", "confirmation", "other"] = Field(
        description="이메일의 주요 의도 (예: complaint=불만, inquiry=문의, confirmation=확인, other=기타)"
    )
    sentiment: Literal["positive", "negative", "neutral"] = Field(description="이메일의 감정 상태")
    summary: str = Field(description="이메일 내용 요약")
    next_action: str = Field(description="에이전트가 수행해야 할 다음 단계 (예: 회신, 확인, 무시 등)")

##③ Agent 생성

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import LLMToolEmulator
from langchain.agents.structured_output import ToolStrategy
from langchain.chat_models import init_chat_model

model = init_chat_model("gpt-5-nano")
tools = [send_email_tool, read_email_tool]

agent = create_agent(
    model=model,
    tools=tools,
    # 핵심: 최종 산출물의 규격을 EmailAnalysis 스키마로 강제합니다.
    response_format=ToolStrategy(EmailAnalysis),
    middleware=[
        # 이메일 도구가 실제 백엔드에 없더라도 가상으로 동작하게 해주는 실습용 미들웨어
        LLMToolEmulator(model="gpt-5-nano"),
    ],
)


##④ Agent 실행

In [ ]:
response = agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "최근 온 메일 확인하고 고객의 의도와 감정, 요약, 그리고 어떤 행동이 필요한지 분석하세요."}
        ]
    }
)

In [ ]:
response

In [ ]:
response["messages"]

In [ ]:
response["structured_response"]

In [ ]:
analysis = response["structured_response"]

In [ ]:
print(f"\n*intent = {analysis.intent}")
print(f"\n*sentiment = {analysis.sentiment}")
print(f"\n*summaty = {analysis.summary}")
print(f"\n*next action = {analysis.next_action}")

##⑤ 후처리 응용

In [ ]:
# Pydantic 객체를 딕셔너리로 변환
data_for_db = analysis.model_dump()

# 구조화된 스키마 데이터를 기반으로 한 자동화 파이프라인 예시
if data_for_db["intent"] == "complaint" and data_for_db["sentiment"] == "negative":
    # 1. CS팀 슬랙 채널에 긴급 알림 전송 (API 호출)
    # 2. Jira 이슈 트래커에 '긴급(High)' 티켓 자동 생성
    print("🚨 [긴급] 불만 접수! CS팀에 즉시 알림을 전송합니다.")
    print(f"요약: {data_for_db['summary']}")
elif data_for_db["intent"] == "inquiry":
    # FAQ 데이터베이스 검색 후 자동 회신 스크립트 실행
    print("ℹ️ 일반 문의 접수. 자동 회신 프로세스를 시작합니다.")
